# Vibe Sewing Lab - Production Notebook

Fluxo revisado para execucao em Google Colab (com GPU) e exportacao para GitHub.
Sem mock: usa apenas provider de LLM real.

## Ordem recomendada
1. Runtime setup (clone opcional + validacao de arquivos)
2. Instalacao de dependencias
3. Inicializacao do app Gradio
4. Exportacao do pacote ZIP

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('GPU (COLAB_GPU):', os.getenv('COLAB_GPU', '0'))

# Se os arquivos nao estiverem no runtime, voce pode definir VIBE_REPO_URL
# Exemplo: os.environ['VIBE_REPO_URL'] = 'https://github.com/<user>/<repo>.git'
repo_url = os.getenv('VIBE_REPO_URL', '').strip()
repo_dir = Path(os.getenv('VIBE_REPO_DIR', 'VibeSwingLab'))

if not Path('vibe_sewing_lab_prod.py').exists() and repo_url:
    if not repo_dir.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', repo_url, str(repo_dir)])
    os.chdir(repo_dir)

cwd = Path.cwd()
print('Working directory:', cwd)

required_files = ['vibe_sewing_lab_prod.py', 'VibeSewing_Production.ipynb']
missing = [name for name in required_files if not (cwd / name).exists()]
if missing:
    raise FileNotFoundError(
        'Arquivos obrigatorios ausentes no runtime: ' + ', '.join(missing) + '. '
        + "Faca upload dos arquivos ou configure VIBE_REPO_URL."
    )

if (cwd / 'requirements.txt').exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    # Fallback para execucao do notebook sem requirements.txt
    packages = [
        'gradio',
        'opencv-python',
        'pillow',
        'numpy',
        'pandas',
        'matplotlib',
        'plotly',
        'faiss-cpu',
        'sentence-transformers',
        'transformers',
        'accelerate',
        'langchain',
        'langchain-community',
        'langchain-text-splitters',
        'ollama',
        'google-generativeai',
        'openai',
    ]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

print('Dependencias instaladas com sucesso.')

In [ ]:
import os

from vibe_sewing_lab_prod import build_app

# Opcional: configure antes de rodar o app
# os.environ['VIBE_LLM_PROVIDER'] = 'qwen'
# os.environ['VIBE_LLM_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'
# os.environ['VIBE_LLM_PROVIDER'] = 'gemini'  # requer GEMINI_API_KEY

app = build_app()
is_colab = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ
share = os.getenv('VIBE_GRADIO_SHARE', '1' if is_colab else '0') == '1'
debug = os.getenv('VIBE_GRADIO_DEBUG', '0') == '1'

try:
    app.launch(share=share, debug=debug)
except Exception:
    app.launch(share=False, debug=debug)

In [ ]:
from pathlib import Path
from zipfile import ZipFile

root = Path.cwd()
zip_path = root / 'vibe_sewing_lab_production_export.zip'
with ZipFile(zip_path, 'w') as zf:
    for name in ['vibe_sewing_lab_prod.py', 'VibeSewing_Production.ipynb', 'requirements.txt']:
        file_path = root / name
        if file_path.exists():
            zf.write(file_path, arcname=name)
print(f'Export gerado em: {zip_path}')